# Module 3 — Classification: Customer Churn
**Zoomcamp ch.3 · Telco churn · started 2026-06-12**

Binary classification: predict whether a Telco customer will churn (`Churn = Yes`).
This is your first *production classifier* workflow — logistic regression, categorical encoding, and reasoning about which features matter.

**How to use this notebook:** it's a scaffold, not a solution. Each section has the *strategic question* to answer and an empty cell to write the code yourself. Reference solution is in `practice/alexey_reference_03-churn.ipynb` — peek only after you've attempted.

**sklearn-vs-lecture flag (your known pattern):** Alexey hand-codes logistic regression sigmoid/training in numpy for intuition. Production = `sklearn.linear_model.LogisticRegression`. They are NOT identical: sklearn applies **L2 regularization by default** (`C=1.0`), Alexey's hand-rolled version doesn't. When numbers disagree, suspect this first (it bit you on HW2 Q4 with Ridge). The headline you should internalize is the **sklearn** one; the numpy is scaffolding for understanding.

## 0 - Frame the problem
*Strategic Qs (answer in one line each):*
- What's the **target**? What's the positive class, and what's the base rate (class balance)?
- Why is **accuracy alone a trap** here? (hint: relate to the base rate you'll compute below)
- Business framing: is a false negative (missed churner) or a false positive (wasted retention offer) more costly? This drives which metric you optimize in ch.4.

## 1 - Get the data

In [19]:
import pandas as pd
import numpy as np

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.shape
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


## 2 - Clean & prepare
Production-grade prep before any modelling. Three known gotchas in this dataset:
1. **Column / string normalization** - lowercase column names and string cells, replace spaces with `_` (Alexey's convention; keeps `df.churn` accessible).
2. **`TotalCharges` is dtype `object`** - 11 rows are blank strings (new customers, tenure 0). Coerce with `pd.to_numeric(..., errors="coerce")` then fill. *Strategic Q: fill with 0 or with the mean? Justify.*
3. **Target encoding** - map `Churn` Yes/No -> integer 1/0 so it's a numeric `y`.

In [20]:
# TODO: normalize columns + string values, fix TotalCharges, encode churn -> 0/1
df.columns = df.columns.str.lower().str.replace(" ", "_")
categorical_cols = list(df.dtypes[df.dtypes == "object"].index)
for col in categorical_cols:
    df[col] = df[col].str.lower().str.replace(" ", "_")
df.columns

df.dtypes

# TotalCharges has some empty strings, which cannot be converted to float. We will replace them with 0 and convert the column to float.
tc = pd.to_numeric(df["totalcharges"], errors="coerce")
df.totalcharges = pd.to_numeric(df["totalcharges"], errors="coerce").fillna(0)
df.totalcharges.isna().sum()

# Encode churn column to true false
df.churn = df.churn.replace({"yes": 1, "no": 0})
# 1. Replace the strings with numbers
df.churn = df.churn.replace({"Yes": 1, "No": 0})

# 2. Convert the column type to integer
df.churn = df.churn.astype(int)

df.churn.head()


0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

## 3 - Set up validation framework
*Do this BEFORE looking at features* - otherwise you leak.
`train_test_split` twice -> 60% train / 20% val / 20% test (`random_state=1`, like the lecture). Separate `y` out of each frame and `del df['churn']` from the feature frames so you never accidentally train on the label.

In [9]:
from sklearn.model_selection import train_test_split

# TODO: 60/20/20 split, extract y_train/y_val/y_test, drop target from feature frames
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# We will further split the full train set into train and validation sets,
# we need 20% of the original data for validation, which is 25% of the full train set.
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

len(df), len(df_train), len(df_val), len(df_test)

# I like to reset the index after splitting, it makes it easier to work with the dataframes later on.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)  
df_test = df_test.reset_index(drop=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values   

del df_train["churn"]
del df_val["churn"]
del df_test["churn"]    

## 4 - EDA & feature importance - *the strategist core of this chapter*
Don't just plot. For each categorical feature, quantify how much it moves churn. Three lenses (compute on the **train set only**):

- **Churn rate per group vs. global rate** - `groupby(feature).churn.mean()`.
- **Risk ratio** = group_rate / global_rate. >1 => higher-risk group; <1 => protective. (Think relative risk, like a hazard multiplier.)
- **Difference** = group_rate - global_rate.

Then rank ALL features by **mutual information** (categorical->target) and **correlation** (numeric->target).
*Strategic Q: which 3 features look most predictive, and does the direction make business sense (e.g. month-to-month contract, no tech support)?*

In [18]:
# TODO: global churn rate; churn rate / risk ratio per categorical;
# mutual_info_score ranking; numeric correlation


df_full_train=df_full_train.reset_index(drop=True)
df_full_train.head()

# let see if there are any missing values in the full train set
df_full_train.isna().sum()

# let's look at the distribution of the target variable
df_full_train.churn.value_counts(normalize=True)



churn
0    0.734469
1    0.265531
Name: proportion, dtype: float64

## 5 - Encode features -> one-hot
`DictVectorizer(sparse=False)` on `df.to_dict(orient="records")` is the lecture's path and the production-friendly one (handles categorical + numeric in one transformer, gives you `.get_feature_names_out()`).
*Strategic Q: why one-hot and not integer-label the categories? (a linear model would invent a fake ordering.)*

In [ ]:
from sklearn.feature_extraction import DictVectorizer

# TODO: fit DV on train dicts, transform train & val


## 6 - Train logistic regression
**Headline (production):** `sklearn.linear_model.LogisticRegression`.
Then (optional, for intuition) compare to Alexey's numpy sigmoid + gradient version.
*Strategic Qs:*
- What does the **sigmoid** do that linear regression's raw output can't? (bounds to (0,1) -> a probability.)
- `model.predict_proba(X)[:,1]` gives churn probability; `0.5` is the default decision threshold. Where would you move it given the FN-vs-FP cost you decided in section 0?

In [ ]:
from sklearn.linear_model import LogisticRegression

# TODO: fit LogisticRegression on train, predict_proba on val
# remember: sklearn defaults to C=1.0 (L2). Alexey's hand-version has no reg -> expect small diffs


## 7 - Evaluate (accuracy for now; full metrics in ch.4)
Accuracy on val, and compare against the **baseline** = always predict the majority class (= 1 - base rate).
*Strategic Q: how much better than baseline are you? If it's only a couple points, what does that tell you?*
Then **interpret**: pull `dict(zip(dv.get_feature_names_out(), model.coef_[0]))`, sort by weight. Positive weight => pushes toward churn. Do the signs match your section 4 intuition?

In [ ]:
from sklearn.metrics import accuracy_score

# TODO: val accuracy vs majority-class baseline; inspect coefficients for interpretation


## 8 - Wrap-up checklist (Sunday review)
- [ ] Can I explain **why accuracy is misleading at ~26.5% churn** without notes?
- [ ] Risk ratio: can I define it and read it off a table?
- [ ] sklearn `LogisticRegression` default reg (`C`) - internalized?
- [ ] One-hot via `DictVectorizer` - why, not just how?
- [ ] **Append a Playbook card** for: logistic regression, one-hot/DictVectorizer, mutual information, risk ratio (see `ML_Strategist_Playbook.md` maintenance rule).

Next: **ch.4 metrics** (`practice/04-metrics` reference) - precision/recall, ROC-AUC, the metrics that actually matter for imbalanced churn.